# Exercicio 1: k-NN com Dataset Cardio

In [1]:
import pandas as pd
from sklearn import neighbors
from sklearn.model_selection import train_test_split

cardio = pd.read_csv('../data/cardio_v2_876b68506f73d316e861b89de59c2808.csv', sep=';')
cardio = cardio[(cardio['ap_hi'] > 0) & (cardio['ap_lo'] > 0)]
cardio.shape

(69969, 13)

In [2]:
cardio.describe()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
count,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000,69969.000000
mean,49973.718733,19468.946162,1.349512,164.362375,74.208325,128.778602,96.663651,1.366862,1.226529,0.088125,0.053781,0.803713,0.499750
std,28851.777613,2467.011396,0.476819,8.247259,14.395338,153.860173,188.505690,0.680240,0.572339,0.283478,0.225587,0.397191,0.500004
min,0.000000,10798.000000,1.000000,55.000000,10.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,25007.000000,17664.000000,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
50%,50002.000000,19703.000000,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,74891.000000,21327.000000,2.000000,170.000000,82.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000
max,99999.000000,23713.000000,2.000000,370.000000,200.000000,16020.000000,11000.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000


In [3]:
cardio.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110.0,80,1,1,0,0,1,0
2,2,18857,1,165,64.0,130.0,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150.0,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100.0,60,1,1,0,0,0,0
5,8,21914,1,151,67.0,120.0,80,2,2,0,0,0,0


## Parte 1: k-NN original (sem IMC)

In [4]:
X = cardio[['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']]
y = cardio['cardio']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print('Treino:', X_train.shape[0])
print('Validacao:', X_val.shape[0])
print('Teste:', X_test.shape[0])

Treino: 55975
Validacao: 6997
Teste: 6997


In [5]:
melhor_k = 0
melhor_acuracia = 0

for k in range(2, 30):
    clf = neighbors.KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    acertos = (y_pred == y_val).sum()
    acuracia = acertos / len(y_val)
    if acuracia > melhor_acuracia:
        melhor_acuracia = acuracia
        melhor_k = k

print('Melhor k:', melhor_k)
print('Acuracia validacao:', melhor_acuracia)

Melhor k: 27
Acuracia validacao: 0.7024439045305131


In [6]:
clf = neighbors.KNeighborsClassifier(n_neighbors=melhor_k)
clf.fit(X_train, y_train)
y_pred_teste = clf.predict(X_test)
acertos_teste = (y_pred_teste == y_test).sum()
acuracia_teste = acertos_teste / len(y_test)

print('Acuracia no teste:', acuracia_teste)
print('Acertos:', acertos_teste, 'de', len(y_test))

Acuracia no teste: 0.7117336001143347
Acertos: 4980 de 6997


## Parte 2: k-NN com IMC substituindo peso e altura

In [7]:
cardio['imc'] = cardio['weight'] / ((cardio['height'] / 100) ** 2)
cardio['imc'].describe()

count    69969.000000
mean        27.557196
std          6.092359
min          3.471784
25%         23.875115
50%         26.377898
75%         30.222222
max        298.666667
Name: imc, dtype: float64

In [8]:
X_imc = cardio[['age', 'gender', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'imc']]
y = cardio['cardio']

X_train_imc, X_temp_imc, y_train_imc, y_temp_imc = train_test_split(X_imc, y, test_size=0.2, random_state=42)
X_val_imc, X_test_imc, y_val_imc, y_test_imc = train_test_split(X_temp_imc, y_temp_imc, test_size=0.5, random_state=42)

print('Treino:', X_train_imc.shape[0])
print('Validacao:', X_val_imc.shape[0])
print('Teste:', X_test_imc.shape[0])

Treino: 55975
Validacao: 6997
Teste: 6997


In [9]:
melhor_k_imc = 0
melhor_acuracia_imc = 0

for k in range(2, 30):
    clf = neighbors.KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train_imc, y_train_imc)
    y_pred = clf.predict(X_val_imc)
    acertos = (y_pred == y_val_imc).sum()
    acuracia = acertos / len(y_val_imc)
    if acuracia > melhor_acuracia_imc:
        melhor_acuracia_imc = acuracia
        melhor_k_imc = k

print('Melhor k:', melhor_k_imc)
print('Acuracia validacao:', melhor_acuracia_imc)

Melhor k: 29
Acuracia validacao: 0.705159354008861


In [10]:
clf = neighbors.KNeighborsClassifier(n_neighbors=melhor_k_imc)
clf.fit(X_train_imc, y_train_imc)
y_pred_teste_imc = clf.predict(X_test_imc)
acertos_teste_imc = (y_pred_teste_imc == y_test_imc).sum()
acuracia_teste_imc = acertos_teste_imc / len(y_test_imc)

print('Acuracia no teste com IMC:', acuracia_teste_imc)
print('Acertos:', acertos_teste_imc, 'de', len(y_test_imc))

Acuracia no teste com IMC: 0.7225953980277262
Acertos: 5056 de 6997


In [11]:
print('Comparacao dos resultados:')
print('Sem IMC - k:', melhor_k, '- Acuracia teste:', acuracia_teste)
print('Com IMC  - k:', melhor_k_imc, '- Acuracia teste:', acuracia_teste_imc)

Comparacao dos resultados:
Sem IMC - k: 27 - Acuracia teste: 0.7117336001143347
Com IMC  - k: 29 - Acuracia teste: 0.7225953980277262


# Exercicio 2: k-NN com Dataset Vinhos Tintos

In [12]:
wine = pd.read_csv('../data/wine/winequality-red.csv', sep=';')
wine.shape

(1599, 12)

In [13]:
wine.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [14]:
wine.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [15]:
print('Distribuicao da qualidade:')
wine.groupby('quality').size()

Distribuicao da qualidade:


quality
3     10
4     53
5    681
6    638
7    199
8     18
dtype: int64

In [16]:
X_wine = wine.drop(columns=['quality'])
y_wine = wine['quality']

X_train_w, X_temp_w, y_train_w, y_temp_w = train_test_split(X_wine, y_wine, test_size=0.2, random_state=42)
X_val_w, X_test_w, y_val_w, y_test_w = train_test_split(X_temp_w, y_temp_w, test_size=0.5, random_state=42)

print('Treino:', X_train_w.shape[0])
print('Validacao:', X_val_w.shape[0])
print('Teste:', X_test_w.shape[0])

Treino: 1279
Validacao: 160
Teste: 160


In [17]:
melhor_k_wine = 0
melhor_acuracia_wine = 0

for k in range(2, 30):
    clf = neighbors.KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train_w, y_train_w)
    y_pred = clf.predict(X_val_w)
    acertos = (y_pred == y_val_w).sum()
    acuracia = acertos / len(y_val_w)
    if acuracia > melhor_acuracia_wine:
        melhor_acuracia_wine = acuracia
        melhor_k_wine = k

print('Melhor k:', melhor_k_wine)
print('Acuracia validacao:', melhor_acuracia_wine)

Melhor k: 27
Acuracia validacao: 0.53125


In [18]:
clf = neighbors.KNeighborsClassifier(n_neighbors=melhor_k_wine)
clf.fit(X_train_w, y_train_w)
y_pred_teste_wine = clf.predict(X_test_w)
acertos_teste_wine = (y_pred_teste_wine == y_test_w).sum()
acuracia_teste_wine = acertos_teste_wine / len(y_test_w)

print('Acuracia no teste:', acuracia_teste_wine)
print('Acertos:', acertos_teste_wine, 'de', len(y_test_w))

Acuracia no teste: 0.50625
Acertos: 81 de 160


In [19]:
print('Predicoes vs Reais (primeiras 20 amostras):')
for i in range(20):
    print('Amostra', i+1, '- Predito:', y_pred_teste_wine[i], '- Real:', y_test_w.iloc[i])

Predicoes vs Reais (primeiras 20 amostras):
Amostra 1 - Predito: 6 - Real: 6
Amostra 2 - Predito: 5 - Real: 5
Amostra 3 - Predito: 6 - Real: 6
Amostra 4 - Predito: 5 - Real: 6
Amostra 5 - Predito: 6 - Real: 6
Amostra 6 - Predito: 6 - Real: 8
Amostra 7 - Predito: 5 - Real: 4
Amostra 8 - Predito: 6 - Real: 5
Amostra 9 - Predito: 6 - Real: 6
Amostra 10 - Predito: 6 - Real: 4
Amostra 11 - Predito: 5 - Real: 7
Amostra 12 - Predito: 5 - Real: 8
Amostra 13 - Predito: 5 - Real: 5
Amostra 14 - Predito: 6 - Real: 5
Amostra 15 - Predito: 6 - Real: 7
Amostra 16 - Predito: 6 - Real: 6
Amostra 17 - Predito: 5 - Real: 6
Amostra 18 - Predito: 5 - Real: 5
Amostra 19 - Predito: 5 - Real: 6
Amostra 20 - Predito: 6 - Real: 6
